In [ ]:
import sys
# add parent directory to the path
sys.path.append("..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torch.autograd import Variable
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, precision_recall_curve, roc_auc_score
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold
import seaborn as sns
from tqdm import tqdm
from timm import create_model
import random
from facenet_pytorch import InceptionResnetV1
from models_vggface.vgg_face import vgg_face, VggFaceFeatureExtractor
import clip
import cv2
from collections import defaultdict

def seed_everything(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # Disable fast algorithms

device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')


In [ ]:
class RareDiseaseDataset(Dataset):
    def __init__(self, df, image_dir, n_way=5, k_shot=1, q_query=1, num_tasks=1000, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.num_tasks = num_tasks
        self.transform = transform
        
        # Group by disease_name and filter
        self.disease_groups = df.groupby('disease_abbr').filter(
            lambda x: len(x) >= k_shot + q_query
        ).groupby('disease_abbr')
        self.diseases = list(self.disease_groups.groups.keys())
        # print("# of classes: ", len(self.diseases))
        
        if len(self.diseases) < n_way:
            raise ValueError(f"Not enough diseases ({len(self.diseases)}) for {n_way}-way task")
    
    def __len__(self):
        return self.num_tasks
    
    def __getitem__(self, idx):
        # seed_everything()
        sampled_diseases = torch.randperm(len(self.diseases))[:self.n_way]
        # print(sampled_diseases)
        sampled_disease_names = [self.diseases[i] for i in sampled_diseases]
        
        support_x = torch.zeros(self.n_way * self.k_shot, 3, 224, 224)
        support_y = torch.zeros(self.n_way * self.k_shot, dtype=torch.long)
        query_x = torch.zeros(self.n_way * self.q_query, 3, 224, 224)
        query_y = torch.zeros(self.n_way * self.q_query, dtype=torch.long)
        
        for i, disease in enumerate(sampled_disease_names):
            disease_df = self.disease_groups.get_group(disease)
            # seed_everything()
            sampled_indices = torch.randperm(len(disease_df))[:self.k_shot + self.q_query]
            # print(sampled_indices)
            
            for j, idx in enumerate(sampled_indices[:self.k_shot]):
                row = disease_df.iloc[idx.item()]
                img_path = os.path.join(self.image_dir[row['synthetic']],
                                        row['disease_abbr'],
                                        row['image_name'])
                img = Image.open(img_path).convert('RGB')
                if self.transform:
                    support_x[i * self.k_shot + j] = self.transform(img)
                support_y[i * self.k_shot + j] = i
            
            for j, idx in enumerate(sampled_indices[self.k_shot:self.k_shot + self.q_query]):
                row = disease_df.iloc[idx.item()]
                img_path = os.path.join(self.image_dir[row['synthetic']],
                        row['disease_abbr'],
                        row['image_name'])
                img = Image.open(img_path).convert('RGB')
                if self.transform:
                    query_x[i * self.q_query + j] = self.transform(img)
                query_y[i * self.q_query + j] = i
        
        return support_x, support_y, query_x, query_y

In [ ]:
class RareDiseaseDatasetEval(Dataset):
    def __init__(self, df, image_dir, n_way=5, k_shot=1, q_query=1, num_tasks=1000, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.num_tasks = num_tasks
        self.transform = transform
        
        # Group by disease_name and filter
        self.disease_groups = df.groupby('disease_abbr').filter(
            lambda x: len(x) >= k_shot + q_query
        ).groupby('disease_abbr')
        self.diseases = list(self.disease_groups.groups.keys())
        # print("# of classes: ", len(self.diseases))
        
        if len(self.diseases) < n_way:
            raise ValueError(f"Not enough diseases ({len(self.diseases)}) for {n_way}-way task")
    
    def __len__(self):
        return self.num_tasks
    
    def __getitem__(self, idx):
        # seed_everything()
        sampled_diseases = torch.randperm(len(self.diseases))[:self.n_way]
        # print(sampled_diseases)
        sampled_disease_names = [self.diseases[i] for i in sampled_diseases]
        
        support_x = torch.zeros(self.n_way * self.k_shot, 3, 224, 224)
        support_y = torch.zeros(self.n_way * self.k_shot, dtype=torch.long)
        query_x = torch.zeros(self.n_way * self.q_query, 3, 224, 224)
        query_y = torch.zeros(self.n_way * self.q_query, dtype=torch.long)
        
        for i, disease in enumerate(sampled_disease_names):
            disease_df = self.disease_groups.get_group(disease)
            # seed_everything()
            sampled_indices = torch.randperm(len(disease_df))[:self.k_shot + self.q_query]
            # print(sampled_indices)
            
            for j, idx in enumerate(sampled_indices[:self.k_shot]):
                img_path = os.path.join(self.image_dir,
                                        disease_df.iloc[idx.item()]['disease_abbr'],
                                        disease_df.iloc[idx.item()]['image_name'])
                img = Image.open(img_path).convert('RGB')
                if self.transform:
                    support_x[i * self.k_shot + j] = self.transform(img)
                support_y[i * self.k_shot + j] = i
            
            for j, idx in enumerate(sampled_indices[self.k_shot:self.k_shot + self.q_query]):
                img_path = os.path.join(self.image_dir,
                                        disease_df.iloc[idx.item()]['disease_abbr'],
                                        disease_df.iloc[idx.item()]['image_name'])
                img = Image.open(img_path).convert('RGB')
                if self.transform:
                    query_x[i * self.q_query + j] = self.transform(img)
                query_y[i * self.q_query + j] = i
        
        return support_x, support_y, query_x, query_y
    

In [ ]:
class ProtoNet(nn.Module):
    def __init__(self, backbone='resnet152'):
        super(ProtoNet, self).__init__()
        self.backbone_name = backbone

        if backbone == 'resnet152':
            base_model = models.resnet152(weights='ResNet152_Weights.IMAGENET1K_V2')
            self.feature_extractor = nn.Sequential(*list(base_model.children())[:-1])
            self.output_dim = base_model.fc.in_features
        
        elif backbone == 'swin_t':
            base_model = models.swin_t(weights="Swin_T_Weights.IMAGENET1K_V1")
            self.feature_extractor = nn.Sequential(
                base_model.features,
                nn.AdaptiveAvgPool2d(1)
            )
            self.output_dim = base_model.head.in_features
        
        elif backbone == 'densenet169':
            base_model = models.densenet169(weights='DenseNet169_Weights.DEFAULT')
            self.feature_extractor = base_model.features
            self.output_dim = base_model.classifier.in_features
        
        elif backbone == 'inception_v3':
            base_model = models.inception_v3(weights='Inception_V3_Weights.DEFAULT')
            self.feature_extractor = nn.Sequential(
                base_model.Conv2d_1a_3x3,
                base_model.Conv2d_2a_3x3,
                base_model.Conv2d_2b_3x3,
                base_model.maxpool1,
                base_model.Conv2d_3b_1x1,
                base_model.Conv2d_4a_3x3,
                base_model.maxpool2,
                base_model.Mixed_5b,
                base_model.Mixed_5c,
                base_model.Mixed_5d,
                base_model.Mixed_6a,
                base_model.Mixed_6b,
                base_model.Mixed_6c,
                base_model.Mixed_6d,
                base_model.Mixed_6e,
                base_model.Mixed_7a,
                base_model.Mixed_7b,
                base_model.Mixed_7c,
                nn.AdaptiveAvgPool2d(1)  # Global Average Pooling
            )
            self.output_dim = base_model.fc.in_features
            
        elif backbone == 'clip':
            self.model, _ = clip.load("ViT-B/32", device="cuda" if torch.cuda.is_available() else "cpu")
            self.model.visual.float()  # Convert CLIP to FP32 to match the input
            self.feature_extractor = self.model.visual
            self.output_dim = 512  # CLIP's ViT-B/32 outputs 512-dimensional features

        # below are pretrained model with vggface
        elif backbone == 'facenet':
            self.feature_extractor = InceptionResnetV1(pretrained='vggface2', classify=False)
            self.output_dim = 512
        elif backbone == 'vggface':
            vggface = vgg_face('models_vggface/vgg_face.pth')
            self.feature_extractor = VggFaceFeatureExtractor(vggface_model=vggface, remove_last_conv=False)
            self.output_dim = 512
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")
        
    def forward(self, x):
        x = self.feature_extractor(x)

        if self.backbone_name in ['clip', 'facenet']:
            x = x / x.norm(dim=-1, keepdim=True) if self.backbone_name == 'clip' else x
        else:
            x = F.adaptive_avg_pool2d(x, (1, 1)).view(x.size(0), -1)  # Global Average Pooling

        return x

In [ ]:
def get_prototypical_logits(model, support_x, support_y, query_x):
    device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
    embeddings_support = model(support_x)  # shape: (N_s, D)
    embeddings_query = model(query_x)      # shape: (N_q, D)

    classes = torch.unique(support_y)
    n_classes = len(classes)

    # Compute prototypes
    prototypes = torch.stack([
        embeddings_support[support_y == cls].mean(dim=0) for cls in classes
    ])  # shape: (n_classes, D)

    # Compute distances to prototypes
    dists = torch.cdist(embeddings_query, prototypes, p=2)  # (N_q, N_classes)
    logits = -dists  # more negative = further away → match CrossEntropyLoss

    return logits

In [ ]:
def train_protonet(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    for support_x, support_y, query_x, query_y in tqdm(dataloader, desc='Training', leave=True):
        support_x, support_y, query_x, query_y = (
            support_x.to(device), support_y.to(device),
            query_x.to(device), query_y.to(device)
        )

        # Flatten the input correctly
        support_x = support_x.view(-1, 3, 224, 224)
        query_x = query_x.view(-1, 3, 224, 224)

        support_y = support_y.view(-1)
        query_y = query_y.view(-1)
        
        logits = get_prototypical_logits(model, support_x, support_y, query_x)
        loss = criterion(logits, query_y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Correct loss and accuracy calculations
        total_loss += loss.item() * query_y.size(0)  # Multiply by batch size
        total_correct += (logits.argmax(dim=1) == query_y).sum().item()
        total_samples += query_y.size(0)

    # Print correct final values outside the loop
    epoch_loss = total_loss / total_samples  # Normalize loss
    epoch_acc = total_correct / total_samples

    return epoch_loss, epoch_acc

def evaluate_protonet(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    with torch.no_grad():
        for support_x, support_y, query_x, query_y in tqdm(dataloader, desc='Evaluating', leave=True):
            support_x, support_y, query_x, query_y = (
                support_x.to(device), support_y.to(device),
                query_x.to(device), query_y.to(device)
            )

            support_x = support_x.view(-1, 3, 224, 224)  
            query_x = query_x.view(-1, 3, 224, 224)  

            support_y = support_y.view(-1)
            query_y = query_y.view(-1)
            
            logits = get_prototypical_logits(model, support_x, support_y, query_x)
            loss = criterion(logits, query_y)

            total_loss += loss.item() * query_y.size(0)  # Accumulate weighted loss
            total_correct += (logits.argmax(dim=1) == query_y).sum().item()
            total_samples += query_y.size(0)

    # Compute final loss and accuracy
    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return avg_loss, accuracy



In [ ]:
def prepare_kfold_dfs(model_name, way, shot, query, k_folds=5, seed=42):
    image_dir_real = "data/rd_images"
    image_dir_syn = "data_aug/synthetic/synthetic_train_realistic"
    csv_path = "data/disease_images.csv"
    data_df = pd.read_csv(csv_path)

    # Define transform
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    if model_name == 'clip':
        transform = clip.load('ViT-B/32')[1]

    # Prepare real disease list (with at least 2 samples)
    disease_groups = data_df.groupby('disease_abbr')
    all_diseases = [d for d in disease_groups.groups if len(disease_groups.get_group(d)) > 1]

    # Split off 20% test diseases
    seed_everything(seed)
    perm = torch.randperm(len(all_diseases)).tolist()
    num_test = int(len(all_diseases) * 0.2)
    test_diseases = [all_diseases[i] for i in perm[:num_test]]
    cv_diseases = [all_diseases[i] for i in perm[num_test:]]

    test_df = data_df[data_df['disease_abbr'].isin(test_diseases)].copy()
    test_df['synthetic'] = False
    test_loader = DataLoader(
        RareDiseaseDatasetEval(test_df, image_dir_real, n_way=way, k_shot=1, q_query=1, num_tasks=200, transform=transform),
        batch_size=1, num_workers=4, shuffle=False
    )

    # Load synthetic metadata
    synthetic_entries = []
    for class_dir in os.listdir(image_dir_syn):
        class_path = os.path.join(image_dir_syn, class_dir)
        if not os.path.isdir(class_path): continue
        for fname in os.listdir(class_path):
            if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                synthetic_entries.append({
                    'image_name': fname,
                    'disease_abbr': class_dir,
                    'synthetic': True
                })
    synthetic_df = pd.DataFrame(synthetic_entries)

    fold_loaders = []
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=seed)

    for fold, (train_idx, val_idx) in enumerate(kf.split(cv_diseases)):
        train_diseases = [cv_diseases[i] for i in train_idx]
        val_diseases = [cv_diseases[i] for i in val_idx]
        
        # For this fold: filter real + synthetic to training diseases only
        train_df = data_df[data_df['disease_abbr'].isin(train_diseases)].copy()
        train_df['synthetic'] = False

        # Count real samples per class
        real_counts = train_df['disease_abbr'].value_counts().to_dict()

        # Filter synthetic only for training classes
        syn_train_df = synthetic_df[synthetic_df['disease_abbr'].isin(train_diseases)].copy()

        # Prepare synthetic samples to fill up to 10 per class
        synthetic_samples = []

        for cls in train_diseases:
            real_count = real_counts.get(cls, 0)
            needed = max(0, 10 - real_count)

            if needed > 0:
                cls_syn = syn_train_df[syn_train_df['disease_abbr'] == cls]
                n_available = len(cls_syn)

                if n_available >= needed:
                    sampled = cls_syn.sample(needed, random_state=seed)
                    synthetic_samples.append(sampled)
                else:
                    print(f"[Warning] Not enough synthetic samples for class '{cls}': needed {needed}, found {n_available}")
                    # Add all available if we want to pad as much as possible
                    synthetic_samples.append(cls_syn)

        # Combine all sampled synthetic entries
        syn_train_df_limited = pd.concat(synthetic_samples, ignore_index=True) if synthetic_samples else pd.DataFrame(columns=syn_train_df.columns)

        # Combine with real data
        combined_train_df = pd.concat([train_df, syn_train_df_limited], ignore_index=True)

        val_df = data_df[data_df['disease_abbr'].isin(val_diseases)].copy()
        val_df['synthetic'] = False

        train_dataset = RareDiseaseDataset(combined_train_df, {
            False: image_dir_real,
            True: image_dir_syn
        }, way, shot, query, num_tasks=600, transform=transform)

        val_dataset = RareDiseaseDatasetEval(val_df, image_dir_real, n_way=way, k_shot=1, q_query=1, num_tasks=100, transform=transform)

        train_loader = DataLoader(train_dataset, batch_size=1, num_workers=4, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=1, num_workers=4, shuffle=False)

        fold_loaders.append((train_loader, val_loader))

    return fold_loaders, test_loader


In [ ]:
def train_kfold(model_name, fold_loaders, test_loader, way=5, shot=1, query=1, device=None):
    num_epochs = 10
    all_fold_acc = []

    save_dir = f"logs_kfold_pn_dream/{model_name}"
    os.makedirs(save_dir, exist_ok=True)

    for fold_idx, (train_loader, val_loader) in enumerate(fold_loaders):
        print(f"\n================= Fold {fold_idx + 1}/{len(fold_loaders)} =================")

        model = ProtoNet(backbone=model_name).to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        best_acc = 0
        acc_list = []

        fold_save_path = os.path.join(save_dir, f"fold{fold_idx + 1}_best_{way}way_{shot}shot.pth")

        for epoch in range(num_epochs):
            print(f"\nEpoch {epoch + 1}/{num_epochs}")
            train_loss, train_acc = train_protonet(model, train_loader, optimizer, criterion, device)
            val_loss, val_acc = evaluate_protonet(model, val_loader, criterion, device)
            acc_list.append(val_acc)

            if val_acc > best_acc:
                best_acc = val_acc
                torch.save(model.state_dict(), fold_save_path)
                print(f"New best model saved for fold {fold_idx + 1} with Val Acc: {best_acc:.4f}")

            print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        all_fold_acc.append(best_acc)

    print(f"\n================= Cross-Validation Summary =================")
    print(f"Average Validation Accuracy across folds: {np.mean(all_fold_acc):.4f}")


In [ ]:
def test_protonet(model_name, fold_id, test_loader, way, shot, query, device):
    print(f"\n===== Testing Best Model from Fold {fold_id} =====")

    # Initialize model
    model = ProtoNet(backbone=model_name).to(device)
    criterion = nn.CrossEntropyLoss()

    # Load saved weights from best model of this fold
    model_path = f"logs_kfold_pn_dream/{model_name}/fold{fold_id}_best_{way}way_{shot}shot.pth"
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model path not found: {model_path}")
    
    model.load_state_dict(torch.load(model_path))
    model.eval()

    # Evaluate
    test_loss, test_acc = evaluate_protonet(model, test_loader, criterion, device)
    print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")
    
    return test_loss, test_acc


In [ ]:
# resnet152
model_name = 'resnet152'
for way in [15]:
    for shot in [1, 5]:
        print(f"Model: {model_name}, Way: {way}, Shot: {shot}")
        device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
        fold_loaders, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=shot, query=shot)
        train_kfold(model_name=model_name, fold_loaders=fold_loaders, test_loader=test_loader, way=way, shot=shot, query=shot, device=device)

In [ ]:
# resnet152 test
model_name = 'resnet152'
for way in [5, 10, 15]:
    for shot in [1, 5]:
        test_acc_all = []
        _, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=shot, query=1)
        for i in range(5):
            print(f"Model: {model_name}, Way: {way}, Shot: {shot}, Fold: {i + 1}")
            device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
            test_loss, test_acc = test_protonet(model_name=model_name, fold_id=i+1, test_loader=test_loader, way=way, shot=shot, query=1, device=device)
            test_acc_all.append(test_acc)
        mean_acc = np.mean(test_acc_all)
        std_acc = np.std(test_acc_all, ddof=1)
        print(f"Way {way} Average Test Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}% (1-sigma)")
        print("==========================================================")


In [ ]:
# densenet169
model_name = 'densenet169'
for way in [15]:
    for shot in [1, 5]:
        print(f"Model: {model_name}, Way: {way}, Shot: {shot}")
        device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
        fold_loaders, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=shot, query=shot)
        train_kfold(model_name=model_name, fold_loaders=fold_loaders, test_loader=test_loader, way=way, shot=shot, query=shot, device=device)

In [ ]:
# densenet169 test
model_name = 'densenet169'
for way in [5, 10, 15]:
    for shot in [1, 5]:
        test_acc_all = []
        _, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=shot, query=1)
        for i in range(5):
            print(f"Model: {model_name}, Way: {way}, Shot: {shot}, Fold: {i + 1}")
            device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
            test_loss, test_acc = test_protonet(model_name=model_name, fold_id=i+1, test_loader=test_loader, way=way, shot=shot, query=1, device=device)
            test_acc_all.append(test_acc)
        mean_acc = np.mean(test_acc_all)
        std_acc = np.std(test_acc_all, ddof=1)

        print(f"Way {way} Average Test Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}% (1-sigma)")
        
        print("==========================================================")


In [ ]:
# swin_t
model_name = 'swin_t'
for way in [5, 10, 15]:
    for shot in [1, 5]:
        print(f"Model: {model_name}, Way: {way}, Shot: {shot}")
        device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
        fold_loaders, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=shot, query=shot)
        train_kfold(model_name=model_name, fold_loaders=fold_loaders, test_loader=test_loader, way=way, shot=shot, query=shot, device=device)

In [ ]:
# swin_t test
model_name = 'swin_t'
for way in [5, 10, 15]:
    for shot in [1, 5]:
        test_acc_all = []
        _, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=shot, query=1)
        for i in range(5):
            print(f"Model: {model_name}, Way: {way}, Shot: {shot}, Fold: {i + 1}")
            device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
            test_loss, test_acc = test_protonet(model_name=model_name, fold_id=i+1, test_loader=test_loader, way=way, shot=shot, query=1, device=device)
            test_acc_all.append(test_acc)
        mean_acc = np.mean(test_acc_all)
        std_acc = np.std(test_acc_all, ddof=1)

        print(f"Way {way} Average Test Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}% (1-sigma)")
        
        print("==========================================================")

In [ ]:
# clip
model_name = 'clip'
for way in [5, 10, 15]:
    for shot in [1, 5]:
        print(f"Model: {model_name}, Way: {way}, Shot: {shot}")
        device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
        fold_loaders, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=shot, query=shot)
        train_kfold(model_name=model_name, fold_loaders=fold_loaders, test_loader=test_loader, way=way, shot=shot, query=shot, device=device)

In [ ]:
# clip test
model_name = 'clip'
for way in [5, 10, 15]:
    for shot in [1, 5]:
        test_acc_all = []
        _, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=shot, query=1)
        for i in range(5):
            print(f"Model: {model_name}, Way: {way}, Shot: {shot}, Fold: {i + 1}")
            device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
            test_loss, test_acc = test_protonet(model_name=model_name, fold_id=i+1, test_loader=test_loader, way=way, shot=shot, query=1, device=device)
            test_acc_all.append(test_acc)
        mean_acc = np.mean(test_acc_all)
        std_acc = np.std(test_acc_all, ddof=1)

        print(f"Way {way} Average Test Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}% (1-sigma)")
        
        print("==========================================================")

In [ ]:
# facenet
model_name = 'facenet'
for way in [5, 10, 15]:
    for shot in [1, 5]:
        print(f"Model: {model_name}, Way: {way}, Shot: {shot}")
        device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
        fold_loaders, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=shot, query=shot)
        train_kfold(model_name=model_name, fold_loaders=fold_loaders, test_loader=test_loader, way=way, shot=shot, query=shot, device=device)

In [ ]:
# facenet test
model_name = 'facenet'
for way in [5, 10, 15]:
    for shot in [1, 5]:
        test_acc_all = []
        _, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=shot, query=1)
        for i in range(5):
            print(f"Model: {model_name}, Way: {way}, Shot: {shot}, Fold: {i + 1}")
            device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
            test_loss, test_acc = test_protonet(model_name=model_name, fold_id=i+1, test_loader=test_loader, way=way, shot=shot, query=1, device=device)
            test_acc_all.append(test_acc)
        mean_acc = np.mean(test_acc_all)
        std_acc = np.std(test_acc_all, ddof=1)

        print(f"Way {way} Average Test Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}% (1-sigma)")
        
        print("==========================================================")

In [ ]:
# vggface
model_name = 'vggface'
for way in [5, 10, 15]:
    for shot in [1, 5]:
        print(f"Model: {model_name}, Way: {way}, Shot: {shot}")
        device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
        fold_loaders, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=shot, query=shot)
        train_kfold(model_name=model_name, fold_loaders=fold_loaders, test_loader=test_loader, way=way, shot=shot, query=shot, device=device)

In [ ]:
# vggface test
model_name = 'vggface'
for way in [5, 10, 15]:
    for shot in [1, 5]:
        test_acc_all = []
        _, test_loader = prepare_kfold_dfs(model_name=model_name, way=way, shot=shot, query=1)
        for i in range(5):
            print(f"Model: {model_name}, Way: {way}, Shot: {shot}, Fold: {i + 1}")
            device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
            test_loss, test_acc = test_protonet(model_name=model_name, fold_id=i+1, test_loader=test_loader, way=way, shot=shot, query=1, device=device)
            test_acc_all.append(test_acc)
        mean_acc = np.mean(test_acc_all)
        std_acc = np.std(test_acc_all, ddof=1)

        print(f"Way {way} Average Test Accuracy: {mean_acc*100:.2f}% ± {std_acc*100:.2f}% (1-sigma)")
        
        print("==========================================================")